### Esercitazione - E02 - Autoencoder

Scarica un dataset di immagini reali, costruisci il tuo autoencoder e allenalo per il task di image denoising!

* Usa il dataset [CIFAR10](https://pytorch.org/vision/stable/generated/torchvision.datasets.CIFAR10.html).

* Le immagini di questo dataset in genere hanno shape (3, 32, 32).

* Costruisci un autoencoder basato su reti convoluzionali!

* Per costruire il task di denoising devi avere due versioni delle immagini: una rumorosa che dai in input al modello e una pulita, con cui andrai a calcolare la loss.

⚠️ Puoi utilizzare blocchi di codice che abbiamo scritto nei notebook precedenti!

In [1]:
# Import

import os
import numpy

import torch
from torch import nn
import torch.nn.functional as F
from torch import optim
from torchvision import transforms
from torch.utils.data import DataLoader

from torchsummary import summary
from torchvision.utils import save_image

from torchvision.datasets import CIFAR10

In [2]:
# Settiamo gli hyperparametri

LR = 1e-4
EPOCHS = 50
BATCH_SIZE = 32

In [3]:
# Trasformazioni delle immagini

def to_img(x):
    x = 0.5 * (x + 1)
    x = x.clamp(0, 1)
    x = x.view(x.size(0), 3, 32, 32)
    return x


img_transform = transforms.Compose([
    transforms.ToTensor(),
    # transforms.Resize((32)),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

In [4]:
# Scarichiamo il dataset e inseriamolo nel dataloader

dataset = CIFAR10(root="./cifar", download=True, transform=img_transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [5]:
# Definiamo la classe dell'Autoencoder

class VAE(nn.Module):
    def __init__(self, input_shape, latent_dim: int, hidden_dim: int, hidden_layers=3, device=None):
        super(VAE, self).__init__()
        self.input_shape = input_shape  # [3, 32, 32]
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim
        self.device = device if device is not None else torch.device("cpu")

        # Encoder: 5 layer convoluzionali
        modules = []
        in_channels = self.input_shape[0]
        for _ in range(hidden_layers):
            modules.append(
                nn.Sequential(
                    nn.Conv2d(in_channels, hidden_dim, kernel_size=3, stride=2, padding=1),
                    nn.BatchNorm2d(hidden_dim),
                    nn.ReLU()
                )
            )
            in_channels = hidden_dim
        self.encoder = nn.Sequential(*modules)

        # Calcolo dinamico della dimensione flatten (sempre su CPU!)
        with torch.no_grad():
            dummy = torch.zeros(1, *self.input_shape)  # su CPU
            encoder_output = self.encoder(dummy)
            self.flattened_dim = encoder_output.view(1, -1).size(1)

        self.fc_mu = nn.Linear(self.flattened_dim, latent_dim)
        self.fc_var = nn.Linear(self.flattened_dim, latent_dim)

        # Decoder
        self.decoder_input = nn.Linear(latent_dim, self.flattened_dim)
        modules = []
        for _ in range(hidden_layers):
            modules.append(
                nn.Sequential(
                    nn.ConvTranspose2d(hidden_dim, hidden_dim, kernel_size=3, stride=2, padding=1, output_padding=1),
                    nn.BatchNorm2d(hidden_dim),
                    nn.ReLU()
                )
            )
        self.decoder = nn.Sequential(*modules)
        # Layer finale per riportare a 3 canali e 32x32
        self.final_layer = nn.Sequential(
            nn.Conv2d(hidden_dim, self.input_shape[0], kernel_size=3, padding=1),
            nn.Tanh()
        )

    def encode(self, x):
        x = self.encoder(x)
        x = torch.flatten(x, start_dim=1)
        mu = self.fc_mu(x)
        logvar = self.fc_var(x)
        return mu, logvar

    def decode(self, z):
        z = self.decoder_input(z)
        # Recupera la shape dopo l'encoder
        with torch.no_grad():
            dummy = torch.zeros(1, *self.input_shape, device=z.device)
            enc_shape = self.encoder(dummy).shape
        z = z.view(-1, self.hidden_dim, enc_shape[2], enc_shape[3])
        z = self.decoder(z)
        z = self.final_layer(z)
        # Crop o resize per garantire [3,32,32]
        if z.shape[2] > 32 or z.shape[3] > 32:
            z = z[:, :, :32, :32]
        elif z.shape[2] < 32 or z.shape[3] < 32:
            z = F.interpolate(z, size=(32, 32), mode='bilinear', align_corners=False)
        return z

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std, device=std.device)
        return eps * std + mu

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        output = self.decode(z)
        return output, mu, logvar


In [6]:
# Definiamo un'istanza dell'Autoencoder
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vae = VAE(dataset[0][0].shape, latent_dim=32, hidden_dim=32, device=device).to(device)

# Definiamo la loss function
reconstruction_function = nn.MSELoss(reduction='sum').to(device)

def to_img(x):
    x = 0.5 * (x + 1)
    x = x.clamp(0, 1)
    x = x.view(x.size(0), 3, 32, 32)
    return x.cpu()  # Per salvataggio immagini, riportiamo su CPU

def loss_function(recon_x, x, mu, logvar):
    """
    recon_x: immagine ricostruita
    x: immagine originale
    mu: media
    logvar: varianza (log)
    """
    recon_loss = reconstruction_function(recon_x, x)  # mse loss
    # loss = 0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    KLD_element = mu.pow(2).add_(logvar.exp()).mul_(-1).add_(1).add_(logvar)
    KLD = torch.sum(KLD_element).mul_(-0.5)
    # KL divergence
    return recon_loss + KLD

# Definiamo l'ottimizzatore
optimizer = optim.Adam(vae.parameters(), lr=1e-3)


In [9]:
verbose = True

def fit():

    for epoch in range(EPOCHS):

        vae.train()
        train_loss = 0

        for batch, data in enumerate(dataloader):

            img, _ = data

            img = img.to(device)

            optimizer.zero_grad()

            new_batch, mu, logvar = vae(img) #forward

            loss = loss_function(new_batch, img, mu, logvar) #loss

            loss.backward()

            train_loss = train_loss + loss.item()

            optimizer.step()

        if verbose:
            print(f"Epoch: {epoch}: Loss = {loss.item()/len(img):.4f}")

        save = to_img(new_batch)
        save_image(save, "./vae_img/image_{}.png".format(epoch))

In [10]:
fit()

Epoch: 0: Loss = 220.1399
Epoch: 1: Loss = 248.6809
Epoch: 1: Loss = 248.6809
Epoch: 2: Loss = 301.4733
Epoch: 2: Loss = 301.4733
Epoch: 3: Loss = 240.0609
Epoch: 3: Loss = 240.0609
Epoch: 4: Loss = 212.6037
Epoch: 4: Loss = 212.6037
Epoch: 5: Loss = 169.8108
Epoch: 5: Loss = 169.8108
Epoch: 6: Loss = 210.1523
Epoch: 6: Loss = 210.1523
Epoch: 7: Loss = 195.7846
Epoch: 7: Loss = 195.7846
Epoch: 8: Loss = 219.2303
Epoch: 8: Loss = 219.2303
Epoch: 9: Loss = 259.8677
Epoch: 9: Loss = 259.8677
Epoch: 10: Loss = 190.6447
Epoch: 10: Loss = 190.6447
Epoch: 11: Loss = 199.1481
Epoch: 11: Loss = 199.1481
Epoch: 12: Loss = 232.4150
Epoch: 12: Loss = 232.4150
Epoch: 13: Loss = 216.1836
Epoch: 13: Loss = 216.1836
Epoch: 14: Loss = 185.3903
Epoch: 14: Loss = 185.3903
Epoch: 15: Loss = 225.5479
Epoch: 15: Loss = 225.5479
Epoch: 16: Loss = 206.9215
Epoch: 16: Loss = 206.9215
Epoch: 17: Loss = 217.5854
Epoch: 17: Loss = 217.5854
Epoch: 18: Loss = 233.4610
Epoch: 18: Loss = 233.4610
Epoch: 19: Loss = 20

In [ ]:
# Scarichiamo il dataset per il test e inseriamolo nel dataloader

In [ ]:
# Testiamo il modello sul test set